# Vectors as Matrix

In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

## Create Chunks from TXT

In [2]:
from google.cloud import storage

client = storage.Client()
bucket = client.get_bucket("experiment-b77ffdf0-1101-44eb-8d3c-3375c185554d")
blob = bucket.blob("experiment-files/cars.txt")
full_text = blob.download_as_text()

# Split into ~500 char chunks with some overlap
chunk_size = 500
overlap = 50
chunks = [
    full_text[i:i+chunk_size] 
    for i in range(0, len(full_text), chunk_size - overlap)
]

print(f"Created {len(chunks)} chunks")

Created 18 chunks


## Embed Chunks

In [3]:
# pip install google-cloud-aiplatform

from vertexai.language_models import TextEmbeddingModel

model = TextEmbeddingModel.from_pretrained("text-embedding-005")

def embed(texts: list[str]) -> list[list[float]]:
    embeddings = model.get_embeddings(texts)
    return [e.values for e in embeddings]

# Example: chunk a document into ~500-char pieces, then embed
chunk_store = {f"chunk_{i}": text for i, text in enumerate(chunks)}
vectors = embed(chunks)  # shape: (2, 768)

## Magic

In [4]:
import numpy as np
import vertexai
from vertexai.generative_models import GenerativeModel

#vertexai.init(project="moodels-educ", location="europe-west6")
vertexai.init(project="moodels-educ", location="us-central1")
gemini = GenerativeModel("gemini-2.5-flash-lite")

def really_ask_gemini(question, context):
    # Generate answer
    prompt = f"""Answer the question using only the context below.

    Context:
    {context}
    
    Question: {question}
    """
    response = gemini.generate_content(prompt)
    return response.text

def ask_question(question): 
    # Embed the question
    [q_vector] = embed([question])

    # Stack all your vectors into a matrix
    vector_matrix = np.array(vectors)  # shape: (num_chunks, 768)
    q = np.array(q_vector)
    
    # Cosine similarity
    scores = vector_matrix @ q / (np.linalg.norm(vector_matrix, axis=1) * np.linalg.norm(q))
    
    # Top 5
    top_k = np.argsort(scores)[::-1][:5]
    
    retrieved_ids = [f"chunk_{i}" for i in top_k]
    context = "\n\n".join(chunk_store[f"chunk_{i}"] for i in top_k)
    
    print("Top chunks:", top_k)
    print("Context preview:", context[:300])
    
    return really_ask_gemini(question, context)

## Question

In [5]:
answer = ask_question("What are the key differences between SUVs and sedans?")
print("\nanswer:\n")
print(answer)

Top chunks: [6 7 0 2 5]
Context preview: nce, efficiency, and aesthetics. Aerodynamics is especially important, as it affects fuel consumption and stability at high speeds. Designers use wind tunnels and simulations to optimize airflow around the vehicle.

Sports cars are designed for performance, speed, and handling. They often feature po

answer:

SUVs prioritize space, comfort, and off-road capability. Sedans offer a balance between comfort and efficiency.
